# Tensor cores: WMMA, MMA and why fast GEMMs need big tiles

What a tensor core does in one instruction, how to drive it from CUDA with the WMMA API, why a naive tensor-core kernel is still memory-bound (the balance point jumps by about 8 times), and how each GPU generation changed the interface, from mma.sync to Hopper's warpgroup wgmma and Blackwell's tensor memory. Ends with when to reach for CUTLASS, Triton or a library instead.

This notebook holds the code from the lesson **[Tensor cores: WMMA, MMA and why fast GEMMs need big tiles](https://inferenceacademy.com/learn/kern-cuda/tensor-cores-wmma-and-mma)** on Inference Academy, in the same order. Read the lesson for the explanations, diagrams and exercises, and run the cells here as you go.

**Before you start:** choose *Runtime → Change runtime type → T4 GPU*, then run the first cell to check the GPU.

In [ ]:
!nvidia-smi

## What a tensor core does

## WMMA: the portable API

Snippet from the lesson (add it to your program; not run on its own):

```cpp
#include <mma.h>
#include <cuda_fp16.h>
using namespace nvcuda;

// A: M x K, B: K x N, both row-major FP16. C: M x N FP32. M, N and K must be multiples of 16.
// Each warp computes one 16 x 16 tile of C.
__global__ void wmma_gemm(const half* A, const half* B, float* C, int M, int N, int K) {
  int warp = (blockIdx.x * blockDim.x + threadIdx.x) / 32;
  int tiles_n = N / 16;
  int tm = warp / tiles_n, tn = warp % tiles_n;
  if (tm * 16 >= M) return;                    // the whole warp leaves together

  wmma::fragment<wmma::matrix_a, 16, 16, 16, half, wmma::row_major> a;
  wmma::fragment<wmma::matrix_b, 16, 16, 16, half, wmma::row_major> b;
  wmma::fragment<wmma::accumulator, 16, 16, 16, float> acc;
  wmma::fill_fragment(acc, 0.f);

  for (int k = 0; k < K; k += 16) {
    wmma::load_matrix_sync(a, A + (size_t)tm * 16 * K + k, K);   // last arg: leading dimension
    wmma::load_matrix_sync(b, B + (size_t)k * N + tn * 16, N);
    wmma::mma_sync(acc, a, b, acc);
  }
  wmma::store_matrix_sync(C + (size_t)tm * 16 * N + tn * 16, acc, N, wmma::mem_row_major);
}
// launch: int warps = (M / 16) * (N / 16); wmma_gemm<<<(warps * 32 + 127) / 128, 128>>>(A, B, C, M, N, K);
```

The next cell uses `wmma.cu`, which you put together yourself from the lesson's snippets (or an earlier lesson). Create it first in a cell that starts with `%%writefile wmma.cu`.

In [ ]:
!nvcc -O3 -arch=sm_75 -o wmma wmma.cu && ./wmma

In [ ]:
import torch, triton
a = torch.randn(4096, 4096, device="cuda", dtype=torch.half)
b = torch.randn(4096, 4096, device="cuda", dtype=torch.half)
ms = triton.testing.do_bench(lambda: a @ b)
print(f"cuBLAS FP16: {2 * 4096**3 / (ms / 1e3) / 1e12:.1f} TFLOP/s")

## Why the simple kernel is slow

## The generations, briefly

## So who writes these kernels?

## What you can now do

## Common mistakes

## Exercise

## Going deeper

---
Code from [Inference Academy](https://inferenceacademy.com), MIT licensed. Lesson: https://inferenceacademy.com/learn/kern-cuda/tensor-cores-wmma-and-mma